# 08c. 의료 QA 챗봇만 실행 (v2.1)

08 노트북에서 **챗봇에 필요한 셀만** 뽑은 버전입니다. 평가 없이 **런타임 → 모두 실행** 한 번이면 됩니다 (A100, 약 5~10분).

- 검색: bge 후보 100개 → MiniLM-L-6 재정렬 상위 5개 (중복 제거) / 생성: Qwen2.5-7B + 06 문장형 LoRA
- 거절 기준 4.92는 08에서 검증 질문으로 정한 값을 그대로 씁니다
- 마지막 셀 출력의 `https://xxxx.gradio.live` 링크를 팀원에게 보내면 됩니다. **셀이 돌고 있는 동안만** 열려 있고, 끝내려면 ■(중지)
- 대화 기록은 드라이브 `rag_msmarco/health/rag/chat_log.jsonl`에 쌓입니다

## 1. 설치 + 드라이브

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft gradio
!pip -q uninstall -y torchao    # Colab 기본 torchao가 최신 peft와 충돌해서 제거 (06과 같음)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR    = '/content/drive/MyDrive/rag_msmarco/health'                 # v2.1 DB (02)
OUT_DIR     = f'{SAVE_DIR}/rag'
REPORT_DIR  = '/content/drive/MyDrive/RAG/2_results/generation_v21'       # 결과 요약 사본 (Mac RAG 폴더와 동기화)
LORA_DIR    = f'{SAVE_DIR}/lora_sentence/adapter'                         # 06 문장형 LoRA
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)
print('06 문장형 LoRA 있음:', os.path.isdir(LORA_DIR))

## 2. 설정

In [ ]:
import re, json, time, random, string
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
N_ANS      = 500        # 평가할 '답 있는 건강 질문' 수 (최대 500)
GEN_BATCH  = 16
MAX_NEW    = 256
CHAT_LORA  = True       # 챗봇에서 06 문장형 LoRA를 쓸지 (False면 원본 Qwen)

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. 검색기 (v2.1 DB 불러오기, 약 2~3분)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 평가 {len(eval_set)} / 검증 {len(val_set)} ({time.time() - t:.0f}초)')
print(Counter((q['domain'], q['answerable']) for q in eval_set))

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

TIERS = {
    '공공기관':   ['medlineplus.gov', 'cdc.gov', 'nih.gov', 'nhs.uk', 'who.int', 'fda.gov', 'cancer.gov'],
    '병원·학회':  ['mayoclinic.org', 'clevelandclinic.org', 'hopkinsmedicine.org', 'aafp.org', 'merckmanuals.com',
                  'kidshealth.org', 'health.harvard.edu', 'heart.org', 'cancer.org', 'familydoctor.org', 'uptodate.com'],
    '상업 매체':  ['webmd.com', 'healthline.com', 'verywell.com', 'medicinenet.com', 'everydayhealth.com', 'drugs.com',
                  'rxlist.com', 'emedicinehealth.com', 'medscape.com', 'medicalnewstoday.com', 'rightdiagnosis.com',
                  'patient.info', 'labtestsonline.org'],
    '커뮤니티·생활': ['healthtap.com', 'sharecare.com', 'livestrong.com', 'babycenter.com', 'psychologytoday.com'],
    '사전':       ['thefreedictionary.com'],
}

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def tier_of(site):
    for tier, sites_ in TIERS.items():
        if any(site == s or site.endswith('.' + s) for s in sites_):
            return tier
    return '기타'

sites = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def retrieve_many(queries, dedup=True, top_n=TOP_N):
    # 질문마다 [{pid, score, top_raw, site, tier, url, text}, ...] 상위 top_n개
    _, ids = index.search(encode(queries), K)
    pairs = [(q, texts[p]) for q, cands in zip(queries, ids.tolist()) for p in cands]
    all_scores = reranker.predict(pairs, batch_size=512, show_progress_bar=len(queries) > 300).reshape(len(queries), K)
    out = []
    for cands, scores in zip(ids.tolist(), all_scores):
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = int(cands[i])
            key = norm_text(texts[p])
            if dedup and key in seen:
                continue
            seen.add(key)
            docs.append({'pid': p, 'score': float(scores[i]), 'top_raw': float(scores.max()), 'site': sites[p],
                         'tier': tier_of(sites[p]), 'url': corpus[p]['url'], 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

retrieve = lambda q, **kw: retrieve_many([q], **kw)[0]

## 4. 거절 기준 (08에서 검증 200개로 정한 값)

In [ ]:
THRESH = 4.921875   # 08: 검증 질문 재정렬 최고 점수 하위 5%
print('거절 기준:', THRESH)

## 5. LLM (Qwen2.5-7B + 06 문장형 LoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from contextlib import nullcontext

tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
HAS_LORA = os.path.isdir(LORA_DIR)
if HAS_LORA:
    from peft import PeftModel
    llm = PeftModel.from_pretrained(llm, LORA_DIR)
llm.eval()
print('LLM 불러옴:', LLM_NAME, '+ LoRA' if HAS_LORA else '(LoRA 없음)', f'/ GPU 메모리 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

REFUSAL = "Sorry, I can't answer that from the medical documents I have."

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

SYS_B0 = f'''You are a medical information assistant.
Answer the question from your own knowledge in 2-4 short sentences.
If the question is not about health or medicine, reply with exactly this sentence: "{REFUSAL}"'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

def b0_messages(q):
    return [{'role': 'system', 'content': SYS_B0}, {'role': 'user', 'content': q}]

@torch.inference_mode()
def generate(msgs_list, lora=False, bs=GEN_BATCH):
    # lora=False면 원본 Qwen, True면 06 문장형 LoRA
    ctx = nullcontext() if (lora and HAS_LORA) else (llm.disable_adapter() if HAS_LORA else nullcontext())
    outs = []
    with ctx:
        for i in range(0, len(msgs_list), bs):
            prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.pad_token_id)
            outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

## 6. 챗봇 실행 (링크가 나오면 팀원에게 공유)

In [ ]:
import gradio as gr

USE_LORA_CHAT = CHAT_LORA and HAS_LORA
EXAMPLES = ['What are the symptoms and causes of conjunctivitis?',
            'How long does the flu last?',
            'What is a normal blood pressure for adults?',
            'Who won the world cup in 2014?']

def answer_one(q):
    t = time.time()
    docs = retrieve(q, dedup=True)
    if docs[0]['top_raw'] < THRESH:
        ans, used = REFUSAL, False
    else:
        ans, used = generate([rag_messages(q, docs)], lora=USE_LORA_CHAT)[0], True
    return ans, docs, used, time.time() - t

def sources_md(docs, used, sec):
    head = f'**근거 문단 상위 {len(docs)}개** · {sec:.1f}초'
    if not used:
        head += f'\n\n⚠️ 재정렬 최고 점수 {docs[0]["top_raw"]:.2f} < 기준 {THRESH:.2f} → 관련 문서가 없다고 보고 답하지 않았어요.'
    parts = [head]
    for i, d in enumerate(docs, 1):
        parts.append(f'**[{i}] {d["tier"]}** · `{d["site"]}` · 점수 {d["score"]:.2f}\n\n> {d["text"]}\n\n[{d["url"]}]({d["url"]})')
    return '\n\n---\n\n'.join(parts)

def chat(message, history):
    message = (message or '').strip()
    if not message:
        return '', history, ''
    ans, docs, used, sec = answer_one(message)
    with open(f'{OUT_DIR}/chat_log.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps({'time': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time() + 9 * 3600)),
                            'question': message, 'answer': ans, 'answered': used, 'lora': USE_LORA_CHAT,
                            'docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in docs]},
                           ensure_ascii=False) + '\n')
    history = history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': ans}]
    return '', history, sources_md(docs, used, sec)

model_label = 'Qwen2.5-7B + 문장형 LoRA' if USE_LORA_CHAT else 'Qwen2.5-7B'
with gr.Blocks(title='의료 QA 챗봇 (RAG 데모, v2.1)') as demo:
    gr.Markdown(f'## 의료 QA 챗봇 · RAG 데모 (v2.1)\nMS MARCO 건강 질문 문단 46만 개에서 근거를 찾아 답합니다 ({model_label}). '
                '**영어로 질문하세요.**')
    with gr.Row():
        with gr.Column(scale=3):
            try:
                bot = gr.Chatbot(type='messages', height=480, label='대화')   # Gradio 5
            except TypeError:
                bot = gr.Chatbot(height=480, label='대화')                    # Gradio 6부터 messages 형식이 기본
            box = gr.Textbox(placeholder='e.g. What causes migraines?', label='질문', autofocus=True)
            gr.Examples(EXAMPLES, inputs=box)
            clear = gr.Button('대화 지우기')
        with gr.Column(scale=2):
            src = gr.Markdown('질문하면 근거 문단이 여기에 나옵니다.')
    gr.Markdown('<small>⚠️ 데모용입니다. 의료 조언이 아니며, 문서는 2016년 무렵 자료라 최신 정보와 다를 수 있습니다.</small>')
    box.submit(chat, [box, bot], [box, bot, src])
    clear.click(lambda: ([], '질문하면 근거 문단이 여기에 나옵니다.'), None, [bot, src])

demo.queue().launch(share=True, prevent_thread_lock=True)
try:
    while True:                       # ■(중지)를 누를 때까지 챗봇을 열어 둠
        time.sleep(1)
except KeyboardInterrupt:
    demo.close()
    print('챗봇을 닫았습니다. 대화 기록:', f'{OUT_DIR}/chat_log.jsonl')